# Default GNN Ablation — Kaggle (GPU)

Runs `ablation/ablation_runner.py` on the **default** dataset.

**How to use**
1. In the notebook sidebar set **Accelerator → GPU** and **Internet → On**.
2. Run the setup cells (clone + install) once.
3. Edit only the **ABLATION PARAMETERS** cell to change what gets swept, then run the last cell.

These parameters mirror the module-level constants in `ablation_runner.py`
(`GNN_TYPES`, `N_LAYERS`, `D_MODELS`, `TOP_KS`, `N_HEADS`, `ATTENTIONS`,
`ABLATION_STEPS`, seed / diffusion constants). We override them in-memory so you
never have to touch the source file.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 2. Get the repo (+ data)

Default: clone the public GitHub repo (data is committed in the repo, ~0.5 GB).

If the repo is **private**, either:
- upload it as a Kaggle Dataset and set `REPO_SOURCE = 'kaggle'` + `KAGGLE_DATASET_DIR`, or
- put a GitHub token in `GIT_TOKEN` below.

In [ ]:
import os, shutil

REPO_SOURCE = 'git'                 # 'git'  or  'kaggle'
GIT_URL     = 'https://github.com/stefromp/thesis_project'
GIT_BRANCH  = 'main'
GIT_TOKEN   = ''                    # optional: personal access token for a private repo

# Only used when REPO_SOURCE == 'kaggle' (repo uploaded as a Kaggle Dataset)
KAGGLE_DATASET_DIR = '/kaggle/input/thesis-project'

REPO_ROOT = '/kaggle/working/thesis_project'

if REPO_SOURCE == 'git':
    if os.path.isdir(REPO_ROOT):
        shutil.rmtree(REPO_ROOT)
    url = GIT_URL
    if GIT_TOKEN:
        url = url.replace('https://', f'https://{GIT_TOKEN}@')
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --branch {GIT_BRANCH} {url} {REPO_ROOT}
elif REPO_SOURCE == 'kaggle':
    # Kaggle input is read-only, so copy it into the writable working dir.
    if os.path.isdir(REPO_ROOT):
        shutil.rmtree(REPO_ROOT)
    shutil.copytree(KAGGLE_DATASET_DIR, REPO_ROOT)
else:
    raise ValueError('REPO_SOURCE must be "git" or "kaggle"')

# The .npy data is hosted as a Kaggle Dataset (attach it via 'Add Input'), NOT
# pulled from the repo's Git-LFS store: once the LFS bandwidth quota is hit the
# clone yields pointer files, so np.load('data/default/y_train.npy') fails.
# ---> Set this to the Kaggle Dataset you created for `default` (cf. california's
#      'california-dataset1'); the assert below prints a clear message if wrong.
KAGGLE_DATA_DIR = '/kaggle/input/datasets/stef16/default-dataset'

assert os.path.isdir(KAGGLE_DATA_DIR), \
    f'default data not found at {KAGGLE_DATA_DIR} — attach the Kaggle Dataset via "Add Input"'
# Sanity-check the payloads are real .npy (magic b'\x93NUMPY'), not placeholder files.
for _f in ('y_train.npy', 'X_num_train.npy'):
    _p = os.path.join(KAGGLE_DATA_DIR, _f)
    assert os.path.exists(_p), f'{_f} missing under {KAGGLE_DATA_DIR}'
    with open(_p, 'rb') as _fh:
        assert _fh.read(6) == b'\x93NUMPY', f'{_f} is not a real .npy file'

# eval_catboost's get_catboost_config() derives the dataset name from the LAST
# path component, so expose the read-only Kaggle data under a writable dir named
# exactly 'default' (else it looks for tuned_models/catboost/default-dataset_cv.json).
DATA_DIR = '/kaggle/working/data_real/default'
os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
if os.path.islink(DATA_DIR):
    os.unlink(DATA_DIR)
elif os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.symlink(KAGGLE_DATA_DIR, DATA_DIR)
print('REPO_ROOT =', REPO_ROOT)
print('DATA_DIR  =', DATA_DIR, '->', os.path.realpath(DATA_DIR))
print(sorted(os.listdir(REPO_ROOT)))

## 3. Install dependencies

We keep Kaggle's pre-installed GPU build of **torch / numpy / pandas / scipy /
scikit-learn** (downgrading them tends to break CUDA) and only add the extra
packages the eval pipeline needs.

We deliberately do **not** install `rtdl` / `libzero` — they are source-only on
recent Python and fail to build, and `ablation_runner` injects lightweight stubs
for the `zero` / `rtdl` modules automatically. Packages are installed unpinned so
pip can pick a wheel, and each on its own line so one failure can't abort the rest.

In [ ]:
import sys, importlib, subprocess

# (pip_name, import_name) for everything the pipeline imports.
DEPS = [
    ('icecream',          'icecream'),
    ('catboost',          'catboost'),
    ('category-encoders', 'category_encoders'),
    ('tomli',             'tomli'),
    ('tomli_w',           'tomli_w'),
    ('pynvml',            'pynvml'),
    ('skorch',            'skorch'),
]

def have(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False

# Only install what's actually missing (Kaggle preinstalls most of these), and
# force a wheel so pip never falls back to a source build (the build failures).
missing = [(pip, mod) for pip, mod in DEPS if not have(mod)]
print('missing:', [m for _, m in missing] or 'none')

for pip_name, mod in missing:
    print(f'>>> installing {pip_name}')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:', pip_name],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        # Fall back to allowing a source build, but show the real error if it fails.
        r2 = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name],
            capture_output=True, text=True,
        )
        if r2.returncode != 0:
            print(f'    !! FAILED {pip_name}\n{(r.stderr or r2.stderr)[-1500:]}')

importlib.invalidate_caches()
print('\nfinal check:')
for _, mod in DEPS:
    print('  ok  ' if have(mod) else '  FAIL', mod)

## 4. Default dataset configuration

Taken from `ablation/run_all_ablation.py` (`ALL_DATASETS['default']`), which is
derived from `exp/default/ddpm_cb_best/config.toml`. You normally don't need to
change this.

In [ ]:
DATASET_NAME = 'default'

DEFAULT_CFG = {
    # data is hosted as a Kaggle Dataset and attached as an input (see DATA_DIR in
    # the setup cell); not pulled from the repo's Git-LFS store.
    'real_data_path':         DATA_DIR,
    'num_numerical_features': 14,
    'num_classes':            2,
    'is_y_cond':              True,
    'rtdl_d_layers':          [256, 1024, 1024, 1024, 1024, 512],
    'num_timesteps':          1000,
    'scheduler':              'cosine',
    'lr':                     0.00046818967784044777,
    'weight_decay':           0.0,
    'batch_size':             4096,
    'num_samples':            76800,
    'sample_batch_size':      10000,
    'train_normalization':    'minmax',
    'eval_normalization':     'minmax',
}
DEFAULT_CFG

## 5. ABLATION PARAMETERS  ← edit this cell each run

These are exactly the constants you'd otherwise edit at the top of
`ablation_runner.py`. The full grid is the Cartesian product of the lists
(GCN/GIN ignore `N_HEADS`, so only the first head value runs for them).

Grid size = `len(GNN_TYPES) * len(N_LAYERS) * len(D_MODELS) * len(TOP_KS) * len(N_HEADS) * len(ATTENTIONS)`
(minus the redundant head combos for gcn/gin).

In [ ]:
# ---- ablation grid (change these) ----
GNN_TYPES  = ['gatv2']   # subset of: gcn | gat | gatv2 | gin
N_LAYERS   = [2]                   # number of blocks
D_MODELS   = [64]                # hidden dim
TOP_KS     = [5]                  # sparsity_top_k (0 = dense)
N_HEADS    = [4]                     # attention heads (gcn/gin ignore)
ATTENTIONS = [False]                 # dense self-attention sublayer on/off

# ---- training / eval budget ----
ABLATION_STEPS         = 20_000
N_GEN_SEEDS            = 5
N_CLF_SEEDS           = 10
ABLATION_NUM_TIMESTEPS = 1000
ABLATION_BATCH_SIZE    = 4096
ABLATION_LR            = 1e-4

# ---- run options ----
DEVICE  = 'cuda:0'
SEED    = 0
NO_SKIP = False   # True = re-run combos even if results already exist

## 6. Run the ablation

Imports the real `ablation_runner`, overwrites its module-level constants with
the values from the cell above, then runs the full grid for default. Results
land in `{REPO_ROOT}/exp/default/ablation/<combo>/results_full_averaged.json`
and a consolidated `ablation_summary.json`.

In [ ]:
import sys, argparse

for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'ablation'), os.path.join(REPO_ROOT, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

# eval_catboost loads tuned_models/catboost/<ds>_cv.json via a RELATIVE path,
# so the process cwd must be the repo root.
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

import ablation_runner as ar

# Override the module constants with the values chosen above.
ar.GNN_TYPES             = GNN_TYPES
ar.N_LAYERS              = N_LAYERS
ar.D_MODELS              = D_MODELS
ar.TOP_KS                = TOP_KS
ar.N_HEADS               = N_HEADS
ar.ATTENTIONS            = ATTENTIONS
ar.ABLATION_STEPS        = ABLATION_STEPS
ar.N_GEN_SEEDS           = N_GEN_SEEDS
ar.N_CLF_SEEDS           = N_CLF_SEEDS
ar.ABLATION_NUM_TIMESTEPS = ABLATION_NUM_TIMESTEPS
ar.ABLATION_BATCH_SIZE   = ABLATION_BATCH_SIZE
ar.ABLATION_LR           = ABLATION_LR

combos = list(ar.iter_ablation_grid())
print(f'Total combinations to run: {len(combos)}')
for c in combos:
    print('  ', ar.exp_dir_name(*c))

args = argparse.Namespace(
    repo_root=REPO_ROOT,
    data_root=REPO_ROOT,
    exp_root=REPO_ROOT,
    device=DEVICE,
    seed=SEED,
    skip_if_done=True,
    no_skip=NO_SKIP,
    aggregate=False,
)

summary = ar.run_ablation(DATASET_NAME, DEFAULT_CFG, args, single_combo=None)

## 7. Inspect / save results

Everything under `exp/default/ablation/` is written into `/kaggle/working`, so
it is downloadable from the notebook's **Output** tab after the session ends.

In [ ]:
import json

summary_path = os.path.join(REPO_ROOT, 'exp', 'default', 'ablation', 'ablation_summary.json')
with open(summary_path) as fh:
    data = json.load(fh)

print('summary:', summary_path, '\n')

def g(metrics, key):
    e = metrics.get(key) if metrics else None
    return f"{e['mean']:.4f}" if isinstance(e, dict) and 'mean' in e else '  -  '

# ML efficiency = macro-F1 (classification) or r2 (regression); plus the three
# TabDDPM privacy/fidelity metrics the run now produces.
hdr = f"{'combo':40s} {'status':8s} {'F1(macro)':>10s} {'DCR_mean':>10s} {'Wass':>8s} {'MIA_AUC':>8s}"
print(hdr); print('-' * len(hdr))
for e in data:
    m = e.get('metrics', {})
    f1 = g(m, 'macro avg/f1-score')
    if f1 == '  -  ':
        f1 = g(m, 'r2')  # regression datasets
    print(f"{e['exp_name']:40s} {e.get('status',''):8s} {f1:>10s} "
          f"{g(m,'dcr_tabddpm_mean'):>10s} {g(m,'wasserstein_mean'):>8s} "
          f"{g(m,'mia_auc'):>8s}")